# WESAD exploratory data analysis (EDA)

This Google Colab notebook explores the prepared WESAD wrist-sensor windows. It checks the data, label and participant balance, sensor feature distributions, correlations, and differences between labels. It also demonstrates a **participant-held-out** split so windows from the same person do not appear in both training and testing.

## Data source
The notebook expects the compact CSV files created by `prepare_wesad.py`, such as `S10_windows.csv`. You can upload the CSV files as a ZIP, upload one or more CSVs directly, or point the notebook at a Google Drive folder. It does not need the original 2.25 GB WESAD ZIP.

Labels in these prepared CSVs: `baseline`, `stress`, and `amusement`. Each row is a 10-second sensor window; columns beginning with `acc_`, `bvp_`, `eda_`, and `temp_` are summary features.

In [ ]:
# Colab dependencies
%pip -q install pandas numpy matplotlib seaborn scikit-learn

## 1. Load the prepared CSV files

Choose one method below. For Drive, set `DATA_DIR` to the folder containing the participant CSV files. For a ZIP, upload a ZIP containing the CSVs (or their folder) when prompted.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

# Option A (recommended for repeat use): read from Google Drive.
# Uncomment these lines, mount Drive, and set the folder path.
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/wesad_windows')

# Option B: upload the CSV files or a ZIP of the CSV files.
DATA_DIR = Path('/content/wesad_windows')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Uncomment to upload files from your computer.
# from google.colab import files
# uploaded = files.upload()
# for name, content in uploaded.items():
#     if name.lower().endswith('.zip'):
#         zip_path = Path('/content') / name
#         zip_path.write_bytes(content)
#         with zipfile.ZipFile(zip_path) as zf:
#             zf.extractall(DATA_DIR)
#     elif name.lower().endswith('.csv'):
#         (DATA_DIR / name).write_bytes(content)

csv_paths = sorted(DATA_DIR.rglob('*_windows.csv'))
if not csv_paths:
    raise FileNotFoundError(
        f'No *_windows.csv files found under {DATA_DIR}. Upload prepared CSVs or set DATA_DIR to their Drive folder.'
    )
df = pd.concat((pd.read_csv(path) for path in csv_paths), ignore_index=True)
print(f'Loaded {len(csv_paths)} files and {len(df):,} windows')
display(df.head())

## 2. Basic structure and data quality

Check types, missing values, duplicate rows, and whether the key columns have sensible values.

In [ ]:
print('Shape:', df.shape)
print('Participants:', df['subject'].nunique())
print('Labels:', sorted(df['label'].dropna().unique()))
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().sort_values(ascending=False).head(15).rename('missing').to_frame())
print('Exact duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)

## 3. Label and participant balance

Class imbalance affects accuracy and model training. Compare counts globally and across participants before choosing metrics or a split.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

label_order = ['baseline', 'stress', 'amusement']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='label', order=label_order, ax=axes[0], color='#4c78a8')
axes[0].set_title('Window count by label')
axes[0].set_xlabel('Label')
sns.countplot(data=df, x='subject', hue='label', hue_order=label_order, ax=axes[1])
axes[1].set_title('Label counts by participant')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_xlabel('Participant')
plt.tight_layout()
plt.show()

display(pd.crosstab(df['subject'], df['label'], normalize='index').round(3))

## 4. Feature distributions by label

The plots below show selected wrist features. Values are summary statistics over each 10-second window. They are useful for exploratory analysis; they do not prove that a feature causes a stress state.

In [ ]:
candidate_features = ['eda_mean', 'bvp_std', 'temp_mean', 'acc_1_std', 'acc_2_std', 'acc_3_std']
features = [c for c in candidate_features if c in df.columns]
if not features:
    raise ValueError('Expected sensor feature columns were not found. Check the CSV schema.')

fig, axes = plt.subplots(len(features), 1, figsize=(10, 4 * len(features)))
if len(features) == 1:
    axes = [axes]
for feature, ax in zip(features, axes):
    sns.boxplot(data=df, x='label', y=feature, order=label_order, showfliers=False, ax=ax)
    ax.set_title(f'{feature} by label')
plt.tight_layout()
plt.show()

## 5. Feature relationships

A correlation matrix can reveal redundant features. Correlation alone does not establish importance or causation.

In [ ]:
numeric_features = [c for c in df.select_dtypes(include='number').columns if c not in ['window', 'label_id']]
corr = df[numeric_features].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='vlag', center=0, square=True)
plt.title('Correlation between numeric sensor features')
plt.tight_layout()
plt.show()

## 6. Compare participants (important for generalization)

Participant-held-out evaluation asks whether a model trained on some people can work for a person it has never seen. The split below reserves one participant for testing. This is more realistic than randomly splitting windows, because windows from one participant share personal sensor and physiology patterns.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in df.columns if c not in ['subject', 'window', 'label', 'label_id']]
X = df[feature_cols].replace([float('inf'), -float('inf')], pd.NA)
X = X.fillna(X.median(numeric_only=True))
y = df['label']
groups = df['subject']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
train_subjects = sorted(groups.iloc[train_idx].unique())
test_subjects = sorted(groups.iloc[test_idx].unique())
print('Train participants:', train_subjects)
print('Test participants:', test_subjects)
print('Train windows:', len(train_idx), '| Test windows:', len(test_idx))
assert set(train_subjects).isdisjoint(test_subjects)

# Fit preprocessing on training data only. This creates scaled arrays ready for a baseline model.
scaler = StandardScaler()
X_train = scaler.fit_transform(X.iloc[train_idx])
X_test = scaler.transform(X.iloc[test_idx])
y_train = y.iloc[train_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()
print('Scaled arrays:', X_train.shape, X_test.shape)

## 7. Optional simple baseline model

This is a quick reference model to confirm the features carry signal. It is an offline logistic regression, not the online-learning method proposed for the project. Report macro F1 and per-class results as well as accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

baseline_model = LogisticRegression(max_iter=1000, class_weight='balanced')
baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=label_order, xticks_rotation=30, cmap='Blues')
plt.title('Participant-held-out logistic regression baseline')
plt.tight_layout()
plt.show()

## Interpretation and limitations

- This EDA uses wrist ACC, BVP, EDA, and temperature summary features from the prepared files. It does not analyze the chest sensor signals.
- The source dataset has only 15 participants, so results may not generalize to a broad population. Repeat group splits or leave-one-subject-out evaluation for research reporting.
- The rows are time windows. For online learning, preserve chronological order within each participant and update only after the correct label/feedback becomes available.
- EDA can guide feature and evaluation choices, but it does not demonstrate that an adaptive interface improves usability. That requires a separate interface study with participants.

### Save a combined EDA-ready table
Optionally combine and save the loaded data in Colab for later modeling.

In [ ]:
output_path = Path('/content/wesad_eda_windows.csv')
df.to_csv(output_path, index=False)
print('Saved:', output_path, '| size:', f'{output_path.stat().st_size:,} bytes')
# In Colab, uncomment to download the combined CSV.
# from google.colab import files
# files.download(str(output_path))